# 02 — KPI Analysis
### Credit Card Customer Intelligence & Churn Analytics

**Goal:** Compute the headline business metrics management would expect on page 1 of an executive dashboard — before any deep-dive charts. These numbers anchor every later section.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

df = pd.read_csv('../data/processed/bankchurners_clean.csv')
print(df.shape)
df.head(2)

(10127, 27)


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Churn_Flag,Avg_Monthly_Spend,Utilization_Bucket,Inactivity_Rate,High_Value_Customer,Engagement_Score
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,"12,691.00",777,"11,914.00",1.33,1144,42,1.62,0.06,0,29.33,Low (0-30%),0.08,No,63.70
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,"8,256.00",864,"7,392.00",1.54,1291,33,3.71,0.10,0,29.34,Low (0-30%),0.08,No,70.20


## 1. Customer base KPIs

In [2]:
total_customers = len(df)
active_customers = (df['Churn_Flag'] == 0).sum()
churned_customers = (df['Churn_Flag'] == 1).sum()
churn_rate = df['Churn_Flag'].mean()

print(f"Total Customers:      {total_customers:,}")
print(f"Active Customers:     {active_customers:,}")
print(f"Churned Customers:    {churned_customers:,}")
print(f"Churn Rate:           {churn_rate:.2%}")

Total Customers:      10,127
Active Customers:     8,500
Churned Customers:    1,627
Churn Rate:           16.07%


## 2. Product / value KPIs

In [3]:
avg_credit_limit = df['Credit_Limit'].mean()
avg_trans_amt = df['Total_Trans_Amt'].mean()
avg_trans_ct = df['Total_Trans_Ct'].mean()
avg_utilization = df['Avg_Utilization_Ratio'].mean()
avg_products = df['Total_Relationship_Count'].mean()
total_trans_volume = df['Total_Trans_Amt'].sum()

print(f"Average Credit Limit:          ${avg_credit_limit:,.0f}")
print(f"Average Transaction Amount:    ${avg_trans_amt:,.0f}")
print(f"Average Transaction Count:     {avg_trans_ct:.1f}")
print(f"Average Utilization Ratio:     {avg_utilization:.1%}")
print(f"Average Products per Customer: {avg_products:.2f}")
print(f"Total Transaction Volume:      ${total_trans_volume:,.0f}")

Average Credit Limit:          $8,632
Average Transaction Amount:    $4,404
Average Transaction Count:     64.9
Average Utilization Ratio:     27.5%
Average Products per Customer: 3.81
Total Transaction Volume:      $44,600,182


**Average Revenue per Customer (proxy):** this dataset has no explicit fee/interest revenue column, so we use `Total_Revolving_Bal` (the balance the bank earns interest on) as the standard proxy for revenue-generating activity per customer — this is the same approach most public analyses of this dataset use, since it's the only column that reflects money the bank is actually carrying/earning on.

In [4]:
avg_revenue_proxy = df['Total_Revolving_Bal'].mean()
print(f"Average Revenue per Customer (revolving balance proxy): ${avg_revenue_proxy:,.0f}")

Average Revenue per Customer (revolving balance proxy): $1,163


## 3. Engagement / risk KPIs

In [5]:
inactive_3plus_pct = (df['Months_Inactive_12_mon'] >= 3).mean()
premium_card_pct = df['Card_Category'].isin(['Gold', 'Platinum']).mean()
high_value_pct = (df['High_Value_Customer'] == 'Yes').mean()
avg_engagement = df['Engagement_Score'].mean()

print(f"Inactive 3+ Months (%):         {inactive_3plus_pct:.1%}")
print(f"Premium Card Holders (%):       {premium_card_pct:.1%}")
print(f"High-Value Customers (%):       {high_value_pct:.1%}")
print(f"Average Engagement Score:       {avg_engagement:.1f} / 100")

Inactive 3+ Months (%):         45.3%
Premium Card Holders (%):       1.3%
High-Value Customers (%):       19.6%
Average Engagement Score:       55.2 / 100


## 4. KPI summary table

A single table suitable for pasting straight onto the dashboard's KPI cards page.

In [6]:
kpi_summary = pd.DataFrame([
    {"KPI": "Total Customers", "Value": f"{total_customers:,}"},
    {"KPI": "Active Customers", "Value": f"{active_customers:,}"},
    {"KPI": "Churned Customers", "Value": f"{churned_customers:,}"},
    {"KPI": "Churn Rate", "Value": f"{churn_rate:.2%}"},
    {"KPI": "Average Credit Limit", "Value": f"${avg_credit_limit:,.0f}"},
    {"KPI": "Average Transaction Amount", "Value": f"${avg_trans_amt:,.0f}"},
    {"KPI": "Average Transaction Count", "Value": f"{avg_trans_ct:.1f}"},
    {"KPI": "Average Utilization Ratio", "Value": f"{avg_utilization:.1%}"},
    {"KPI": "Average Products per Customer", "Value": f"{avg_products:.2f}"},
    {"KPI": "Total Transaction Volume", "Value": f"${total_trans_volume:,.0f}"},
    {"KPI": "Avg Revenue per Customer (proxy)", "Value": f"${avg_revenue_proxy:,.0f}"},
    {"KPI": "Inactive 3+ Months (%)", "Value": f"{inactive_3plus_pct:.1%}"},
    {"KPI": "Premium Card Holders (%)", "Value": f"{premium_card_pct:.1%}"},
    {"KPI": "High-Value Customers (%)", "Value": f"{high_value_pct:.1%}"},
    {"KPI": "Average Engagement Score", "Value": f"{avg_engagement:.1f} / 100"},
])
kpi_summary

,KPI,Value
0,Total Customers,"10,127"
1,Active Customers,"8,500"
2,Churned Customers,"1,627"
3,Churn Rate,16.07%
4,Average Credit Limit,"$8,632"
5,Average Transaction Amount,"$4,404"
6,Average Transaction Count,64.9
7,Average Utilization Ratio,27.5%
8,Average Products per Customer,3.81
9,Total Transaction Volume,"$44,600,182"


## 5. KPIs split by churn status

The real business question isn't just "what's the average" — it's "how do churned customers differ from retained ones?" This is the first hint of *why* people are leaving, ahead of the full EDA in notebook 03.

In [7]:
compare_cols = ['Credit_Limit', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Avg_Utilization_Ratio',
                'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon',
                'Total_Revolving_Bal', 'Engagement_Score']

comparison = df.groupby('Attrition_Flag')[compare_cols].mean().T
comparison['% Difference'] = (
    (comparison['Attrited Customer'] - comparison['Existing Customer']) / comparison['Existing Customer']
) * 100
comparison

Attrition_Flag,Attrited Customer,Existing Customer,% Difference
Credit_Limit,"8,136.04","8,726.88",-6.77
Total_Trans_Amt,"3,095.03","4,654.66",-33.51
Total_Trans_Ct,44.93,68.67,-34.57
Avg_Utilization_Ratio,0.16,0.30,-45.19
Total_Relationship_Count,3.28,3.91,-16.22
Months_Inactive_12_mon,2.69,2.27,18.45
Contacts_Count_12_mon,2.97,2.36,26.14
Total_Revolving_Bal,672.82,"1,256.60",-46.46
Engagement_Score,40.71,57.93,-29.72


**Early read (to be confirmed with full EDA + stats in notebook 03):** churned customers tend to show fewer products held, more inactive months, more service contacts, and lower transaction activity — consistent with the classic disengagement-before-churn pattern the business hypothesis in the brief calls out.

## 6. Save KPI outputs

In [8]:
kpi_summary.to_csv('../data/processed/kpi_summary.csv', index=False)
comparison.to_csv('../data/processed/kpi_churn_comparison.csv')
print("Saved kpi_summary.csv and kpi_churn_comparison.csv")

Saved kpi_summary.csv and kpi_churn_comparison.csv
